# Pebble v1 — Stage 1: MLM pre-training + isolation ablation

Run the blocks top-to-bottom. State carries across cells, so each block uses what the previous one defined. Plan ref: `improvement-plan-from-deep-read.md` 1.3 / D-F.

## 0. Install pinned NeoBERT stack  (run once, ~4 min)

In [ ]:
# Cell 1 — pinned NeoBERT stack (Kaggle default torch 2.10 drops sm_60 -> P100 crash).
import os
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"
get_ipython().system('pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 xformers==0.0.28.post3 --index-url https://download.pytorch.org/whl/cu121')
get_ipython().system('pip install -q transformers==4.48.2 sentencepiece')
print("install cell done")


## 1. Imports & config

In [ ]:
# Block 1 — imports, seed, config
import os, re, math, random, copy, warnings, urllib.request
warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset as TorchDataset
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import f1_score
from datasets import load_dataset
from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL    = "chandar-lab/NeoBERT"
REVISION = "5424c8efeea6491b151d62dee55a752165407430"
MAX_LEN, BATCH = 64, 32
MLM_EPOCHS, MLM_MASK_PROB, MLM_CORPUS_CAP = 2, 0.30, 12000
FT_EPOCHS, FT_PER_POOL = 3, 2500
EMO_VAL_N, SEV_VAL_N = 1000, 600
EIREG_NEG = {"anger", "fear", "sadness"}
EIREG_EMOS = ["anger", "fear", "joy", "sadness"]
ART = "/kaggle/working"
print("config ready | device", DEVICE)


## 2. Tokenizer & data  (GoEmotions -> emotion, SemEval EI-reg -> severity)

In [ ]:
# Block 2 — tokenizer + data (GoEmotions -> emotion, SemEval EI-reg -> severity)
tok = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, trust_remote_code=True)
VOCAB = tok.vocab_size
print("vocab", VOCAB, "| mask_token", tok.mask_token, tok.mask_token_id)

def eireg(file_split):
    base = "https://raw.githubusercontent.com/cbaziotis/ntua-slp-semeval2018/master/datasets/task1/EI-reg"
    rows = []
    for emo in EIREG_EMOS:
        url = f"{base}/EI-reg-En-{emo}-{file_split}.txt"
        try:
            raw = urllib.request.urlopen(url, timeout=40).read().decode("utf-8")
        except Exception as e:
            print("eireg download failed:", url, e); return None
        for ln in raw.splitlines()[1:]:
            c = ln.split("\t")
            if len(c) < 4: continue
            try: inten = float(c[3])
            except ValueError: continue
            rows.append({"text": c[1], "severity": inten if emo in EIREG_NEG else 0.0})
    return rows

go_train = load_dataset("go_emotions", "simplified", split="train")
go_val   = load_dataset("go_emotions", "simplified", split="validation")
EMO_NAMES = go_train.features["labels"].feature.names
NEUTRAL = EMO_NAMES.index("neutral")
N_EMO = len(EMO_NAMES)
def go_rows(ds):
    return [{"text": r["text"], "emotion": (r["labels"][0] if r["labels"] else NEUTRAL)} for r in ds]
go_tr, go_va = go_rows(go_train), go_rows(go_val)

ei_tr, ei_va = eireg("train"), eireg("dev")
if not ei_tr or not ei_va:
    print("!! eireg unavailable -> synthetic severity fallback")
    rnd = lambda: random.random()
    ei_tr = [{"text": f"i feel low energy and distress sample {i} {rnd():.2f}", "severity": rnd()} for i in range(3000)]
    ei_va = [{"text": f"a tough day sample {i} {rnd():.2f}", "severity": rnd()} for i in range(600)]
print(f"GoEmotions train={len(go_tr)} val={len(go_va)} ({N_EMO} classes) | EI-reg train={len(ei_tr)} dev={len(ei_va)}")

def encode(texts):
    e = tok(texts, truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
    return e["input_ids"], e["attention_mask"]
print("data ready")


## 3. MLM pre-training — 30% masking on in-domain text
Watch the loss drop across epochs.

In [ ]:
# Block 3 — MLM pre-training: 30% masking on in-domain text. Watch the loss drop.
corpus = [r["text"] for r in go_tr] + [r["text"] for r in ei_tr]
random.shuffle(corpus); corpus = corpus[:MLM_CORPUS_CAP]
mlm_ids, mlm_attn = encode(corpus)
print(f"[MLM] corpus={len(corpus)} texts, {int(MLM_MASK_PROB*100)}% masking, {MLM_EPOCHS} epochs")

SPECIAL = torch.tensor(tok.all_special_ids)
def mask_batch(ids):
    ids = ids.clone(); labels = ids.clone()
    keep = torch.isin(ids, SPECIAL)
    prob = torch.full(ids.shape, MLM_MASK_PROB); prob[keep] = 0.0
    sel = torch.bernoulli(prob).bool()
    labels[~sel] = -100
    r = torch.rand(ids.shape)
    ids[sel & (r < 0.8)] = tok.mask_token_id
    rnd_pos = sel & (r >= 0.8) & (r < 0.9)
    ids[rnd_pos] = torch.randint(VOCAB, ids.shape)[rnd_pos]
    return ids, labels

# NeoBERTLMHead: .model is the inner encoder; forward -> MaskedLMOutput(logits), no loss.
mlm = AutoModelForMaskedLM.from_pretrained(MODEL, revision=REVISION, trust_remote_code=True).to(DEVICE)
enc_ref = mlm.model
opt = torch.optim.AdamW(mlm.parameters(), lr=5e-5, weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()
order = list(range(len(corpus)))
mlm.train()
for ep in range(MLM_EPOCHS):
    random.shuffle(order); tot = 0.0; nb = 0
    for i in range(0, len(order), BATCH):
        idx = order[i:i + BATCH]
        ids, labels = mask_batch(mlm_ids[idx])
        ids, labels, attn = ids.to(DEVICE), labels.to(DEVICE), mlm_attn[idx].to(DEVICE)
        opt.zero_grad()
        with torch.cuda.amp.autocast():
            logits = mlm(input_ids=ids, attention_mask=attn).logits
            loss = F.cross_entropy(logits.view(-1, VOCAB), labels.view(-1), ignore_index=-100)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item(); nb += 1
    print(f"  [MLM] epoch {ep+1}/{MLM_EPOCHS}  loss={tot/nb:.4f}")
print("[MLM] training done")


## 4. Save the adapted encoder  (the reusable artifact)

In [ ]:
# Block 4 — save the adapted encoder (the reusable artifact), then free MLM memory.
adapted_state = {k: v.detach().half().cpu() for k, v in enc_ref.state_dict().items()}
torch.save(adapted_state, f"{ART}/mlm_encoder.pt")
print(f"[MLM] saved adapted encoder -> {ART}/mlm_encoder.pt ({len(adapted_state)} tensors, fp16)")
del mlm, enc_ref, opt; torch.cuda.empty_cache()
print("memory freed")


## 5. Fine-tune setup — masked two-pool multi-task

In [ ]:
# Block 5 — fine-tune setup: masked two-pool data, model, metrics, finetune() fn.
EMO, SEV = 0, 1   # per-example task id; the mask decides which head contributes
def build_records(emo_rows, sev_rows):
    recs  = [{"text": r["text"], "task": EMO, "emo": r["emotion"], "sev": 0.0} for r in emo_rows]
    recs += [{"text": r["text"], "task": SEV, "emo": -1, "sev": r["severity"]} for r in sev_rows]
    return recs
random.shuffle(go_tr); random.shuffle(ei_tr)
train_recs = build_records(go_tr[:FT_PER_POOL], ei_tr[:FT_PER_POOL]); random.shuffle(train_recs)

class FTDataset(TorchDataset):
    def __init__(self, recs):
        self.ids, self.attn = encode([r["text"] for r in recs])
        self.task = torch.tensor([r["task"] for r in recs])
        self.emo  = torch.tensor([r["emo"] for r in recs], dtype=torch.long)
        self.sev  = torch.tensor([r["sev"] for r in recs], dtype=torch.float)
    def __len__(self): return len(self.task)
    def __getitem__(self, i): return self.ids[i], self.attn[i], self.task[i], self.emo[i], self.sev[i]
train_loader = DataLoader(FTDataset(train_recs), batch_size=BATCH, shuffle=True)
emo_val_ids, emo_val_attn = encode([r["text"] for r in go_va[:EMO_VAL_N]])
emo_val_y = np.array([r["emotion"] for r in go_va[:EMO_VAL_N]])
sev_val_ids, sev_val_attn = encode([r["text"] for r in ei_va[:SEV_VAL_N]])
sev_val_y = np.array([r["severity"] for r in ei_va[:SEV_VAL_N]], dtype=float)

class Head(nn.Module):
    def __init__(s, h, out, d=256, p=0.1):
        super().__init__()
        s.net = nn.Sequential(nn.Dropout(p), nn.Linear(h, d), nn.GELU(), nn.Dropout(p), nn.Linear(d, out))
    def forward(s, x): return s.net(x)
class MultiTask(nn.Module):
    def __init__(s, adapted=None):
        super().__init__()
        s.encoder = AutoModel.from_pretrained(MODEL, revision=REVISION, trust_remote_code=True)
        if adapted is not None:
            s.encoder.load_state_dict({k: v.float() for k, v in adapted.items()})
        h = getattr(s.encoder.config, "hidden_size", 768)
        s.emotion_head, s.score_head = Head(h, N_EMO), Head(h, 1)
    def forward(s, ids, attn):
        cls = s.encoder(input_ids=ids, attention_mask=attn).last_hidden_state[:, 0, :]
        return s.emotion_head(cls), torch.sigmoid(s.score_head(cls)).squeeze(-1)

# metrics (improvement-plan section 4)
def pearson(p, t):  return 0.0 if np.std(p) < 1e-8 or np.std(t) < 1e-8 else float(pearsonr(p, t)[0])
def spearman(p, t): return 0.0 if np.std(p) < 1e-8 or np.std(t) < 1e-8 else float(spearmanr(p, t)[0])
def ece(probs, correct, n=10):
    conf = probs.max(1); b = np.linspace(0, 1, n + 1); e = 0.0; N = len(conf)
    for i in range(n):
        m = (conf > b[i]) & (conf <= b[i + 1])
        if m.sum(): e += m.sum() / N * abs(correct[m].mean() - conf[m].mean())
    return float(e)

@torch.no_grad()
def predict(model, ids, attn):
    model.eval(); out_e, out_s = [], []
    for i in range(0, len(ids), BATCH):
        with torch.cuda.amp.autocast():
            el, sp = model(ids[i:i+BATCH].to(DEVICE), attn[i:i+BATCH].to(DEVICE))
        out_e.append(torch.softmax(el.float(), -1).cpu().numpy()); out_s.append(sp.float().cpu().numpy())
    return np.concatenate(out_e), np.concatenate(out_s)

def finetune(tag, adapted):
    set_seed(SEED)
    model = MultiTask(adapted).to(DEVICE)
    opt = torch.optim.AdamW([
        {"params": [p for n, p in model.named_parameters() if not n.startswith("encoder.")], "lr": 2e-5},
        {"params": model.encoder.parameters(), "lr": 1e-5},
    ], weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler()
    for ep in range(FT_EPOCHS):
        model.train()
        for ids, attn, task, emo, sev in train_loader:
            ids, attn, task = ids.to(DEVICE), attn.to(DEVICE), task.to(DEVICE)
            emo, sev = emo.to(DEVICE), sev.to(DEVICE)
            me, ms = task == EMO, task == SEV
            opt.zero_grad()
            with torch.cuda.amp.autocast():
                el, sp = model(ids, attn)
                loss = el.sum() * 0.0                      # keep graph dtype
                if me.any(): loss = loss + F.cross_entropy(el[me], emo[me])
                if ms.any(): loss = loss + F.mse_loss(sp[ms], sev[ms])   # uniform-sum (18's null)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        print(f"  [{tag}] epoch {ep+1}/{FT_EPOCHS} done")
    ep_probs, _ = predict(model, emo_val_ids, emo_val_attn)
    _, sv_pred  = predict(model, sev_val_ids, sev_val_attn)
    pe = ep_probs.argmax(1)
    res = {"arm": tag,
           "emo_macroF1": round(float(f1_score(emo_val_y, pe, average="macro")), 4),
           "emo_ece":     round(ece(ep_probs, (pe == emo_val_y).astype(float)), 4),
           "sev_pearson": round(pearson(sv_pred, sev_val_y), 4),
           "sev_spearman":round(spearman(sv_pred, sev_val_y), 4),
           "sev_mae":     round(float(np.mean(np.abs(sv_pred - sev_val_y))), 4)}
    del model, opt; torch.cuda.empty_cache()
    return res
print("[FT] setup ready:", len(train_recs), "train records")


## 6. Ablation arm A — MLM-OFF (vanilla NeoBERT)

In [ ]:
# Block 6 — ABLATION arm A: MLM-OFF (vanilla NeoBERT encoder)
off = finetune("MLM-off (vanilla)", None)
print("RESULT:", off)


## 7. Ablation arm B — MLM-ON (adapted encoder)

In [ ]:
# Block 7 — ABLATION arm B: MLM-ON (30%-mask domain-adapted encoder)
on = finetune("MLM-on (adapted)", adapted_state)
print("RESULT:", on)


## 8. Results table

In [ ]:
# Block 8 — results table + save
import pandas as pd
df = pd.DataFrame([off, on])
delta = {"arm": "delta (on - off)"}
for c in ["emo_macroF1", "emo_ece", "sev_pearson", "sev_spearman", "sev_mae"]:
    delta[c] = round(on[c] - off[c], 4)
df = pd.concat([df, pd.DataFrame([delta])], ignore_index=True)
df.to_csv(f"{ART}/results_mlm_ablation.csv", index=False)
print("============ MLM ISOLATION ABLATION ============")
print(df.to_string(index=False))
print(f"\nartifacts -> {ART}/mlm_encoder.pt , {ART}/results_mlm_ablation.csv")
print("=== RESULT: SUCCESS ===")
